# 🔍 PAS Silver Layer — Data Exploration & Profiling

**Project**: Global Loom  
**Lakehouse**: `The_Global_Loom`  
**Purpose**: Profile all 18 silver tables copied from PAS to understand schema, quality, relationships, and grain before designing the gold layer star schema.

### How to Use
1. Attach this notebook to `The_Global_Loom` lakehouse
2. Run each cell sequentially
3. Share the output of **Cells 2, 4, 8, 10** with your AI assistant (schema, counts, relationships, transaction deep dive)

> ⚠️ Do NOT share raw data values — share schemas, counts, and distributions only.

In [ ]:
# ============================================================
# Cell 1: Setup & Configuration
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import Row
from collections import defaultdict

LAKEHOUSE = "The_Global_Loom"

# ── Table Registry ──
# Adjust these if your table names differ after copying
REF_TABLES = [
    "CarrierHierarchy",
    "FinancialGeographyHierarchy",
    "FinancialSegmentHierarchy",
    "IndustryHierarchy",
]

RPT_TABLES = [
    "vwAddress",
    "vwCarrierHierarchy",
    "vwCFInvoice",
    "vwCFParty",
    "vwDataSourceInstance",
    "vwFinancialGeography",
    "vwParty",
    "vwPolicy",
    "vwPolicyLayer",
    "vwPolicyPartyRole",
    "vwProduct",
    "vwTransaction",
    "vwTransactionDetailUSD",
    "vwTransactionSummaryUSD",
]

ALL_TABLES = REF_TABLES + RPT_TABLES

def safe_read(table_name):
    """Read a table with error handling. Returns DataFrame or None."""
    try:
        return spark.sql(f"SELECT * FROM {LAKEHOUSE}.{table_name}")
    except Exception as e:
        print(f"❌ Failed to read {table_name}: {e}")
        return None

print(f"✅ Setup complete. {len(ALL_TABLES)} tables registered.")

## Phase 1: Table Discovery
List all tables available in the lakehouse to verify names and accessibility.

In [ ]:
# ============================================================
# Cell 2: Discover available tables
# ============================================================
print("📋 Tables available in the lakehouse:")
print("=" * 60)
available = spark.sql(f"SHOW TABLES IN {LAKEHOUSE}")
display(available)

## Phase 2: Table Overview
Row counts and column counts for every table. This tells us which tables are large (fact candidates) vs small (dimension candidates).

In [ ]:
# ============================================================
# Cell 3: Table Overview — Row Counts & Column Counts
# ============================================================
results = []
for table in ALL_TABLES:
    df = safe_read(table)
    if df:
        row_count = df.count()
        col_count = len(df.columns)
        schema_type = "ref" if table in REF_TABLES else "rpt"
        results.append(Row(Schema=schema_type, Table=table, RowCount=row_count, ColumnCount=col_count))
        print(f"✅ {schema_type}.{table}: {row_count:,} rows × {col_count} cols")
    else:
        results.append(Row(Schema="?", Table=table, RowCount=-1, ColumnCount=-1))

overview_df = spark.createDataFrame(results)
print("\n" + "=" * 60)
print("📊 SUMMARY — ordered by row count (largest first)")
print("=" * 60)
display(overview_df.orderBy(F.desc("RowCount")))

## Phase 3: Schema Inspection
Print column names and data types for every table. **This is the most important cell** — share this output with your AI assistant.

In [ ]:
# ============================================================
# Cell 4: Schema Inspection — Column Names & Types
# ============================================================
for table in ALL_TABLES:
    df = safe_read(table)
    if df:
        print(f"\n{'=' * 60}")
        print(f"📋 {table} ({len(df.columns)} columns)")
        print(f"{'=' * 60}")
        df.printSchema()

## Phase 4: Null & Data Quality Analysis
Calculate null percentage per column. High null rates may indicate optional fields or data quality issues.

In [ ]:
# ============================================================
# Cell 5: Null Percentage Analysis
# ============================================================
for table in ALL_TABLES:
    df = safe_read(table)
    if df is None:
        continue
    total = df.count()
    if total == 0:
        print(f"⚠️ {table} is empty, skipping.")
        continue

    print(f"\n{'=' * 60}")
    print(f"🔍 {table} — Null Analysis ({total:,} rows)")
    print(f"{'=' * 60}")

    null_exprs = [
        F.round(
            (F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) / F.lit(total)) * 100, 1
        ).alias(c)
        for c in df.columns
    ]
    null_pcts = df.select(null_exprs).collect()[0]

    has_nulls = False
    for c in df.columns:
        pct = null_pcts[c]
        if pct and pct > 0:
            bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
            print(f"  {c:<40} {pct:>6.1f}% null  {bar}")
            has_nulls = True

    if not has_nulls:
        print("  ✅ No null values found!")

## Phase 5: Cardinality Analysis
Distinct value counts per column — identifies unique keys (high cardinality) vs dimension candidates (low cardinality).

In [ ]:
# ============================================================
# Cell 6: Cardinality — Distinct Counts Per Column
# ============================================================
for table in ALL_TABLES:
    df = safe_read(table)
    if df is None:
        continue
    total = df.count()
    if total == 0:
        continue

    print(f"\n{'=' * 60}")
    print(f"🔢 {table} — Cardinality ({total:,} rows)")
    print(f"{'=' * 60}")

    distinct_exprs = [F.countDistinct(F.col(c)).alias(c) for c in df.columns]
    distinct_counts = df.select(distinct_exprs).collect()[0]

    for c in df.columns:
        dist = distinct_counts[c]
        ratio = dist / total if total > 0 else 0
        if ratio > 0.95:
            tag = "🔑 UNIQUE KEY"
        elif ratio > 0.5:
            tag = "📊 HIGH"
        elif dist > 20:
            tag = "📋 MEDIUM"
        elif dist > 1:
            tag = "🏷️  LOW (dim candidate)"
        else:
            tag = "⚡ CONSTANT"
        print(f"  {c:<40} {dist:>10,} distinct  ({ratio:.1%})  {tag}")

## Phase 6: Date Column Detection & Ranges
Find all date/timestamp columns and their min/max values. Critical for designing `Dim_Date`.

In [ ]:
# ============================================================
# Cell 7: Date Column Ranges
# ============================================================
for table in ALL_TABLES:
    df = safe_read(table)
    if df is None:
        continue

    date_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, (DateType, TimestampType))
    ]
    if not date_cols:
        continue

    print(f"\n{'=' * 60}")
    print(f"📅 {table} — Date Columns")
    print(f"{'=' * 60}")

    for col_name in date_cols:
        stats = df.select(
            F.min(F.col(col_name)).alias("min_date"),
            F.max(F.col(col_name)).alias("max_date"),
            F.countDistinct(F.col(col_name)).alias("distinct_dates"),
            F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias("null_count")
        ).collect()[0]
        print(f"  {col_name}:")
        print(f"    Range: {stats['min_date']} → {stats['max_date']}")
        print(f"    Distinct: {stats['distinct_dates']:,}  |  Nulls: {stats['null_count']:,}")

## Phase 7: Relationship Discovery
Find column names shared across multiple tables — these are potential join keys for the star schema.

In [ ]:
# ============================================================
# Cell 8: Shared Columns Across Tables (FK Candidates)
# ============================================================
column_to_tables = defaultdict(list)
for table in ALL_TABLES:
    df = safe_read(table)
    if df is None:
        continue
    for col_name in df.columns:
        column_to_tables[col_name].append(table)

shared = {k: v for k, v in column_to_tables.items() if len(v) >= 2}
shared_sorted = sorted(shared.items(), key=lambda x: len(x[1]), reverse=True)

print("🔗 Shared Columns Across Tables (potential FK relationships)")
print("=" * 70)
for col_name, tables in shared_sorted:
    print(f"\n  📌 {col_name} (appears in {len(tables)} tables):")
    for t in tables:
        print(f"     └── {t}")

## Phase 8: Sample Data
First 5 rows of each table for visual inspection. **Review but do NOT share externally.**

In [ ]:
# ============================================================
# Cell 9: Sample Data (first 5 rows per table)
# ============================================================
for table in ALL_TABLES:
    df = safe_read(table)
    if df is None:
        continue
    print(f"\n{'=' * 60}")
    print(f"👁️ {table} — Sample (5 rows)")
    print(f"{'=' * 60}")
    display(df.limit(5))

## Phase 9: Transaction Table Deep Dive
Detailed profiling of the three transaction tables — the primary fact table candidates.

In [ ]:
# ============================================================
# Cell 10: Transaction Tables — Deep Dive
# ============================================================
TXN_TABLES = ["vwTransaction", "vwTransactionDetailUSD", "vwTransactionSummaryUSD"]

for table in TXN_TABLES:
    df = safe_read(table)
    if df is None:
        continue
    total = df.count()
    print(f"\n{'=' * 70}")
    print(f"💰 {table} — Deep Dive ({total:,} rows)")
    print(f"{'=' * 70}")

    # Numeric columns — basic stats
    numeric_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, (IntegerType, LongType, FloatType, DoubleType, DecimalType))
    ]
    if numeric_cols:
        print(f"\n  📊 Numeric Column Statistics:")
        display(df.select(numeric_cols).describe())

    # String columns with low cardinality — value distributions
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for col_name in string_cols:
        distinct_count = df.select(F.countDistinct(col_name)).collect()[0][0]
        if 1 < distinct_count <= 25:
            print(f"\n  🏷️  {col_name} — Value Distribution ({distinct_count} values):")
            display(
                df.groupBy(col_name)
                .agg(F.count("*").alias("Count"))
                .orderBy(F.desc("Count"))
            )

## ✅ Next Steps

After running this notebook, share the following with your AI assistant:

| Cell | What to Share | Why |
|------|--------------|-----|
| Cell 3 | Table overview (row/col counts) | Identify fact vs dim candidates |
| Cell 4 | Schemas (column names + types) | Design gold layer columns |
| Cell 8 | Shared columns | Map FK relationships |
| Cell 10 | Transaction deep dive | Design fact table grain |

> ⚠️ Share **schemas, counts, and distributions only** — never raw data values.

### Decision Points for Star Schema Design
1. **Grain**: What does one row represent in each transaction table?
2. **Which transaction table is the primary fact?** (vwTransaction vs vwTransactionDetailUSD vs vwTransactionSummaryUSD)
3. **Party roles**: How does vwPolicyPartyRole link policies to parties? (bridge table?)
4. **Hierarchy tables**: Do the `ref` hierarchy tables duplicate the `rpt` views? (e.g., CarrierHierarchy vs vwCarrierHierarchy)
5. **Date columns**: Which dates need role-playing relationships? (Transaction date, effective date, expiry date)